In [ ]:
import os
import time
import warnings
from typing import Optional

import numpy as np
import pandas as pd

from arch import arch_model
from joblib import Parallel, delayed
from scipy.stats import skew, kurtosis
from sklearn.cluster import KMeans
from sklearn.model_selection import GroupKFold
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.arima.model import ARIMA

warnings.filterwarnings("ignore")

INPUT_LEN = 480
TARGET_LEN = 120
TOTAL_LEN = 600

EPSILON = 1e-5
RET_CLIP = 0.05

PROFILE_COLS = ["rv", "mean_ret", "skew", "kurt", "max_dd"]

N_CLUSTERS = 10
VAL_RATIO = 0.15
TEST_RATIO = 0.15
SEED = 42

SPEC = dict(ar=1, ma=1, p=1, q=1, dist="normal")

KNN_KS = [3, 5, 10, 20, 40, 80]
CV_FOLDS = 5

ARIMA_MAXITER = 50
GARCH_MAXITER = 50

CACHE_DIR = "arma_garch_knn_cache"
FORCE_REBUILD_FEATURES = False

# Set this to a lower num for a quicker test run. None for the everything.
DEV_N_TIME_IDS: Optional[int] = None
N_JOBS = 8


In [18]:
# =============================================================================
# HELPER FUNCTIONS (SIMILAR TO LSTM)
# =============================================================================

def _safe_log_return_matrix(mat: np.ndarray) -> np.ndarray:
    safe = np.where(mat > 0, mat, 1e-6).astype(np.float32)
    ret = np.zeros_like(safe)
    ret[1:] = np.log(safe[1:] / safe[:-1])
    np.clip(ret, -RET_CLIP, RET_CLIP, out=ret)
    np.nan_to_num(
        ret,
        copy=False,
        nan=0.0,
        posinf=RET_CLIP,
        neginf=-RET_CLIP,
    )
    return ret


def robust_fill(mat: np.ndarray) -> np.ndarray:
    bad = ~np.isfinite(mat) | (mat <= 0)
    if not bad.any():
        return mat
    out = mat.copy()
    rows = out.shape[0]
    row_idx = np.arange(rows)[:, None]
    fwd = np.where(~bad, row_idx, -1)
    np.maximum.accumulate(fwd, axis=0, out=fwd)
    rev = np.where(~bad, row_idx, rows)
    bwd = rows - 1 - np.maximum.accumulate(
        (rows - 1 - rev)[::-1],
        axis=0,
    )[::-1]
    use = np.where(fwd >= 0, fwd, bwd)
    all_bad = (use < 0) | (use >= rows)
    use = np.where(all_bad, 0, use)
    out = out[use, np.arange(out.shape[1])]
    return np.where(all_bad, 1.0, out)


def rmspe(y_true: np.ndarray, y_pred: np.ndarray, eps: float = 1e-8) -> float:
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    y_true = np.where(np.isfinite(y_true), y_true, eps)
    y_pred = np.where(np.isfinite(y_pred), y_pred, 0.0)
    pct = (y_true - y_pred) / (np.abs(y_true) + eps)
    return float(np.sqrt(np.mean(pct ** 2))) * 100.0


def weighted_rmspe(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    weights: np.ndarray,
    eps: float = 1e-8,
) -> float:
    
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    weights = np.asarray(weights, dtype=np.float64)
    y_true = np.where(np.isfinite(y_true), y_true, eps)
    y_pred = np.where(np.isfinite(y_pred), y_pred, 0.0)
    pct = (y_true - y_pred) / (np.abs(y_true) + eps)
    return float(np.sqrt(np.average(pct ** 2, weights=weights))) * 100.0


def mse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    if mask.sum() == 0:
        return np.nan
    return float(np.mean((y_true[mask] - y_pred[mask]) ** 2))


def weighted_mse(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    weights: np.ndarray,
) -> float:
    
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    weights = np.asarray(weights, dtype=np.float64)

    mask = (
        np.isfinite(y_true)
        & np.isfinite(y_pred)
        & np.isfinite(weights)
        & (weights > 0)
    )
    if mask.sum() == 0:
        return np.nan
    return float(
        np.average(
            (y_true[mask] - y_pred[mask]) ** 2,
            weights=weights[mask],
        )
    )


def qlike(y_true: np.ndarray, y_pred: np.ndarray, eps: float = 1e-12) -> float:
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    y_true = np.maximum(y_true, eps)
    y_pred = np.maximum(y_pred, eps)
    ratio = y_true / y_pred
    loss = ratio - np.log(ratio) - 1.0
    loss = loss[np.isfinite(loss)]
    if len(loss) == 0:
        return np.nan
    return float(np.mean(loss))


def weighted_qlike(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    weights: np.ndarray,
    eps: float = 1e-12,
) -> float:
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    weights = np.asarray(weights, dtype=np.float64)
    y_true = np.maximum(y_true, eps)
    y_pred = np.maximum(y_pred, eps)
    ratio = y_true / y_pred
    loss = ratio - np.log(ratio) - 1.0

    mask = (
        np.isfinite(loss)
        & np.isfinite(weights)
        & (weights > 0)
    )
    if mask.sum() == 0:
        return np.nan
    return float(np.average(loss[mask], weights=weights[mask]))


def safe_skew(x: np.ndarray) -> float:
    if np.std(x) <= 1e-12:
        return 0.0
    val = skew(x, bias=False)
    if not np.isfinite(val):
        return 0.0
    return float(val)


def safe_kurt(x: np.ndarray) -> float:
    if np.std(x) <= 1e-12:
        return 0.0
    val = kurtosis(x, bias=False, fisher=True)
    if not np.isfinite(val):
        return 0.0
    return float(val)


def safe_autocorr1(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=np.float64)
    if len(x) < 3:
        return 0.0
    a = x[:-1]
    b = x[1:]
    if np.std(a) <= 1e-12 or np.std(b) <= 1e-12:
        return 0.0
    val = np.corrcoef(a, b)[0, 1]
    if not np.isfinite(val):
        return 0.0
    return float(val)


def _vectorised_window_stats(ret_mat: np.ndarray) -> np.ndarray:
    n_stocks = ret_mat.shape[1]
    out = np.zeros((n_stocks, 5), dtype=np.float32)
    out[:, 0] = np.sqrt(np.einsum("ij,ij->j", ret_mat, ret_mat))
    out[:, 1] = ret_mat.mean(axis=0)
    std_col = ret_mat.std(axis=0)
    nontrivial = std_col > 1e-12

    if nontrivial.any():
        out[nontrivial, 2] = skew(
            ret_mat[:, nontrivial],
            axis=0,
            bias=False,
        )

        out[nontrivial, 3] = kurtosis(
            ret_mat[:, nontrivial],
            axis=0,
            bias=False,
            fisher=True,
        )
    cum_ret = np.exp(np.cumsum(ret_mat, axis=0))
    running_max = np.maximum.accumulate(cum_ret, axis=0)

    out[:, 4] = (
        (running_max - cum_ret) / (running_max + EPSILON)
    ).max(axis=0)

    np.nan_to_num(
        out,
        copy=False,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )
    return out

In [19]:
# =============================================================================
# SPLITTING THE DATA AND CLUSTERING
# =============================================================================
def make_splits(
    df: pd.DataFrame,
    val_ratio: float = 0.15,
    test_ratio: float = 0.15,
    seed: int = 42,
) -> dict:
    rng = np.random.default_rng(seed)
    tids = df["time_id"].unique().copy()
    rng.shuffle(tids)
    n = len(tids)
    n_test = int(n * test_ratio)
    n_val = int(n * val_ratio)
    return {
        "split_summary": {
            "train": {"ids": tids[n_test + n_val:]},
            "val": {"ids": tids[n_test:n_test + n_val]},
            "test": {"ids": tids[:n_test]},
        }
    }


def build_stock_profiles(
    df: pd.DataFrame,
    stock_cols: list,
    train_ids: np.ndarray,
) -> pd.DataFrame:
    print("Step 1, building stock profiles on train only")

    df_s = df[df["time_id"].isin(train_ids)].sort_values(
        ["time_id", "seconds_in_bucket"],
    )
    stat_sum = np.zeros(
        (len(stock_cols), len(PROFILE_COLS)),
        dtype=np.float64,
    )
    stat_count = 0

    for _, grp in df_s.groupby("time_id", sort=False):
        mat = robust_fill(grp[stock_cols].values.astype(np.float32))

        if mat.shape[0] < INPUT_LEN:
            continue

        ret_mat = _safe_log_return_matrix(mat[:INPUT_LEN])

        stat_sum += _vectorised_window_stats(ret_mat).astype(np.float64)
        stat_count += 1

    profiles = pd.DataFrame(
        (stat_sum / max(stat_count, 1)).astype(np.float32),
        index=stock_cols,
        columns=PROFILE_COLS,
    )
    print(f"  Profiles: {profiles.shape} | train time_ids used: {stat_count}")
    return profiles


def cluster_stocks_kmeans(
    profiles: pd.DataFrame,
    n_clusters: int = 10,
    seed: int = 42,
) -> pd.Series:
    print(f"Step 2, KMeans clustering with k={n_clusters}")

    X = StandardScaler().fit_transform(profiles.values)

    km = KMeans(
        n_clusters=n_clusters,
        n_init=20,
        max_iter=500,
        random_state=seed,
    )
    labels = km.fit_predict(X)
    cluster_map = pd.Series(
        labels,
        index=profiles.index,
        name="cluster_id",
    )

    print(
        "  Cluster sizes:",
        cluster_map.value_counts().sort_index().to_dict(),
    )

    return cluster_map


def get_or_build_cluster_map(
    df: pd.DataFrame,
    stock_cols: list,
    train_ids: np.ndarray,
    n_clusters: int,
    seed: int,
) -> tuple[pd.DataFrame, pd.Series]:
    os.makedirs(CACHE_DIR, exist_ok=True)

    profiles_path = os.path.join(
        CACHE_DIR,
        f"profiles_seed{seed}_k{n_clusters}.pkl",
    )

    cluster_path = os.path.join(
        CACHE_DIR,
        f"cluster_map_seed{seed}_k{n_clusters}.pkl",
    )

    if os.path.exists(profiles_path) and os.path.exists(cluster_path):
        profiles = pd.read_pickle(profiles_path)
        cluster_map = pd.read_pickle(cluster_path)

        print("Loaded cached profiles and cluster map")
        print(
            "  Cluster sizes:",
            cluster_map.value_counts().sort_index().to_dict(),
        )
        return profiles, cluster_map

    profiles = build_stock_profiles(
        df=df,
        stock_cols=stock_cols,
        train_ids=train_ids,
    )

    cluster_map = cluster_stocks_kmeans(
        profiles=profiles,
        n_clusters=n_clusters,
        seed=seed,
    )

    profiles.to_pickle(profiles_path)
    cluster_map.to_pickle(cluster_path)

    return profiles, cluster_map

In [20]:
# =============================================================================
# FEATURE EXTRACTION FRROM ARMA GARCH
# =============================================================================

def make_feature_names(n_clusters: int) -> list[str]:
    names = [
        "log_rv_input",
        "log_rv_last120",
        "log_rv_scaled",
        "rv_last120_to_input",
        "rv_scaled_to_input",
        "cluster_mean_return",
        "cluster_std_return",
        "cluster_mean_abs_return",
        "cluster_skew_return",
        "cluster_kurt_return",
        "cluster_max_abs_return",
        "cluster_autocorr1",
        "cluster_size_scaled",
    ]

    names.extend([f"cluster_onehot_{i}" for i in range(n_clusters)])

    names.extend(
        [
            "garch_forecast_rv",
            "garch_forecast_step_vol",
            "garch_forecast_to_input",
            "cond_vol_last",
            "cond_vol_mean",
            "cond_vol_std",
            "omega",
            "alpha_1",
            "beta_1",
            "alpha_plus_beta",
            "arma_ar_1",
            "arma_ma_1",
            "arma_sigma2",
            "resid_std",
            "fit_ok",
        ]
    )

    return names


def get_arma_param(fit, name: str, default: float = 0.0) -> float:
    try:
        names = list(getattr(fit, "param_names", []))
        vals = np.asarray(fit.params, dtype=np.float64)

        if name in names:
            idx = names.index(name)
            val = float(vals[idx])

            if np.isfinite(val):
                return val

    except Exception:
        pass

    return default


def base_feature_vector(task: dict) -> list[float]:
    r = np.asarray(task["inp_ret"], dtype=np.float64)

    onehot = [0.0] * task["n_clusters"]
    onehot[int(task["cluster_id"])] = 1.0

    return [
        float(np.log(task["rv_in"] + EPSILON)),
        float(np.log(task["rv_last120"] + EPSILON)),
        float(np.log(task["rv_scaled"] + EPSILON)),
        float(task["rv_last120"] / (task["rv_in"] + EPSILON)),
        float(task["rv_scaled"] / (task["rv_in"] + EPSILON)),
        float(np.mean(r)),
        float(np.std(r)),
        float(np.mean(np.abs(r))),
        safe_skew(r),
        safe_kurt(r),
        float(np.max(np.abs(r))),
        safe_autocorr1(r),
        float(task["cluster_size"] / max(task["n_stocks"], 1)),
    ] + onehot


def extract_armagarch_feature_row(task: dict, spec: dict) -> dict:
    scale = 100.0

    inp_ret = np.asarray(task["inp_ret"], dtype=np.float64)

    base = base_feature_vector(task)

    fallback_rv = float(task["rv_scaled"])
    fallback_step_vol = fallback_rv / np.sqrt(TARGET_LEN)

    fit_features = [
        fallback_rv,
        fallback_step_vol,
        fallback_rv / (task["rv_in"] + EPSILON),
        fallback_step_vol,
        fallback_step_vol,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        float(np.var(inp_ret * scale)),
        float(np.std(inp_ret * scale)),
        0.0,
    ]

    fit_ok = 0.0

    try:
        if len(inp_ret) < INPUT_LEN:
            raise ValueError("short input")

        if not np.isfinite(inp_ret).all():
            raise ValueError("bad input")

        if np.std(inp_ret) <= 1e-12:
            raise ValueError("flat input")

        y = inp_ret * scale

        try:
            arma_fit = ARIMA(
                y,
                order=(spec["ar"], 0, spec["ma"]),
                trend="c",
                enforce_stationarity=False,
                enforce_invertibility=False,
            ).fit(
                method_kwargs={"maxiter": ARIMA_MAXITER},
            )

        except TypeError:
            arma_fit = ARIMA(
                y,
                order=(spec["ar"], 0, spec["ma"]),
                trend="c",
                enforce_stationarity=False,
                enforce_invertibility=False,
            ).fit()

        resid = np.asarray(arma_fit.resid, dtype=np.float64)

        burn = max(spec["ar"], spec["ma"], 1)
        resid = resid[burn:]
        resid = resid[np.isfinite(resid)]

        if len(resid) < 50:
            raise ValueError("short residual")

        resid = resid - np.mean(resid)

        if np.std(resid) <= 1e-12:
            raise ValueError("flat residual")

        garch = arch_model(
            resid,
            mean="Zero",
            vol="Garch",
            p=spec["p"],
            q=spec["q"],
            dist=spec["dist"],
            rescale=False,
        )

        garch_fit = garch.fit(
            disp="off",
            show_warning=False,
            update_freq=0,
            tol=1e-4,
            options={"maxiter": GARCH_MAXITER},
        )

        fc = garch_fit.forecast(
            horizon=TARGET_LEN,
            reindex=False,
        )

        var_fc = np.asarray(
            fc.variance.values.flatten(),
            dtype=np.float64,
        )

        var_fc = var_fc[np.isfinite(var_fc)]
        var_fc = np.maximum(var_fc, 0.0)

        if len(var_fc) == 0:
            raise ValueError("bad forecast")

        garch_forecast_rv = float(np.sqrt(np.sum(var_fc))) / scale
        garch_forecast_step_vol = float(np.sqrt(np.mean(var_fc))) / scale

        cond_vol = np.asarray(
            garch_fit.conditional_volatility,
            dtype=np.float64,
        )

        cond_vol = cond_vol[np.isfinite(cond_vol)] / scale

        if len(cond_vol) == 0:
            raise ValueError("bad conditional vol")

        params = garch_fit.params

        omega = float(params.get("omega", 0.0))
        alpha_1 = float(params.get("alpha[1]", 0.0))
        beta_1 = float(params.get("beta[1]", 0.0))

        arma_ar_1 = get_arma_param(arma_fit, "ar.L1", 0.0)
        arma_ma_1 = get_arma_param(arma_fit, "ma.L1", 0.0)
        arma_sigma2 = get_arma_param(
            arma_fit,
            "sigma2",
            float(np.var(resid)),
        )

        fit_features = [
            garch_forecast_rv,
            garch_forecast_step_vol,
            garch_forecast_rv / (task["rv_in"] + EPSILON),
            float(cond_vol[-1]),
            float(np.mean(cond_vol)),
            float(np.std(cond_vol)),
            omega,
            alpha_1,
            beta_1,
            alpha_1 + beta_1,
            arma_ar_1,
            arma_ma_1,
            arma_sigma2,
            float(np.std(resid)),
            1.0,
        ]

        fit_ok = 1.0

    except Exception:
        pass

    x = np.array(base + fit_features, dtype=np.float32)

    np.nan_to_num(
        x,
        copy=False,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )

    return {
        "x": x,
        "y_log": float(task["y_log"]),
        "rv_in": float(task["rv_in"]),
        "rv_fut": float(task["rv_fut"]),
        "rv_last120": float(task["rv_last120"]),
        "rv_scaled": float(task["rv_scaled"]),
        "time_id": int(task["time_id"]),
        "cluster_id": int(task["cluster_id"]),
        "cluster_size": int(task["cluster_size"]),
        "fit_ok": fit_ok,
    }


def build_feature_tasks(
    df: pd.DataFrame,
    stock_cols: list,
    cluster_map: pd.Series,
    time_ids: np.ndarray,
    split_name: str,
    dev_n_time_ids: Optional[int] = None,
) -> list[dict]:
    if dev_n_time_ids is not None:
        time_ids = time_ids[:dev_n_time_ids]

    cluster_ids = sorted(cluster_map.unique())
    n_clusters = len(cluster_ids)
    n_stocks = len(stock_cols)

    cluster_stock_idx = {
        cid: np.array(
            [
                stock_cols.index(stock)
                for stock in cluster_map.index[cluster_map == cid]
            ],
            dtype=np.int32,
        )
        for cid in cluster_ids
    }

    cluster_sizes = {
        cid: len(cluster_stock_idx[cid])
        for cid in cluster_ids
    }

    df_sub = df[df["time_id"].isin(time_ids)].sort_values(
        ["time_id", "seconds_in_bucket"],
    )

    tasks = []

    for tid, grp in df_sub.groupby("time_id", sort=True):
        mat = robust_fill(grp[stock_cols].values.astype(np.float32))

        if mat.shape[0] < TOTAL_LEN:
            continue

        ret_mat = _safe_log_return_matrix(mat)

        for cid in cluster_ids:
            idx = cluster_stock_idx[cid]

            r_in_stocks = ret_mat[:INPUT_LEN, idx]
            r_fut_stocks = ret_mat[INPUT_LEN:, idx]

            inp_ret = r_in_stocks.mean(axis=1).astype(np.float64)

            rv_in = float(
                np.sqrt(
                    np.einsum("ij,ij->j", r_in_stocks, r_in_stocks)
                ).mean()
            ) + EPSILON

            rv_fut = float(
                np.sqrt(
                    np.einsum("ij,ij->j", r_fut_stocks, r_fut_stocks)
                ).mean()
            ) + EPSILON

            r_last = r_in_stocks[-TARGET_LEN:]

            rv_last120 = float(
                np.sqrt(
                    np.einsum("ij,ij->j", r_last, r_last)
                ).mean()
            ) + EPSILON

            rv_scaled = float(
                rv_in * np.sqrt(TARGET_LEN / INPUT_LEN)
            ) + EPSILON

            y_log = float(np.log(rv_fut / rv_in))

            tasks.append(
                {
                    "time_id": int(tid),
                    "cluster_id": int(cid),
                    "cluster_size": int(cluster_sizes[cid]),
                    "n_clusters": int(n_clusters),
                    "n_stocks": int(n_stocks),
                    "inp_ret": inp_ret,
                    "rv_in": rv_in,
                    "rv_fut": rv_fut,
                    "rv_last120": rv_last120,
                    "rv_scaled": rv_scaled,
                    "y_log": y_log,
                }
            )

    print(
        f"  {split_name}: built {len(tasks):,} tasks "
        f"from {len(set(t['time_id'] for t in tasks)):,} time_ids"
    )

    return tasks


def feature_cache_path(split_name: str, spec: dict) -> str:
    dev_tag = "full" if DEV_N_TIME_IDS is None else f"dev{DEV_N_TIME_IDS}"

    return os.path.join(
        CACHE_DIR,
        (
            f"features_{split_name}_{dev_tag}"
            f"_k{N_CLUSTERS}"
            f"_seed{SEED}"
            f"_arma{spec['ar']}{spec['ma']}"
            f"_garch{spec['p']}{spec['q']}"
            f"_{spec['dist']}.npz"
        ),
    )


def build_or_load_features(
    df: pd.DataFrame,
    stock_cols: list,
    cluster_map: pd.Series,
    time_ids: np.ndarray,
    split_name: str,
    spec: dict,
    n_jobs: int,
) -> dict:
    os.makedirs(CACHE_DIR, exist_ok=True)

    path = feature_cache_path(split_name, spec)
    feature_names = make_feature_names(N_CLUSTERS)

    if os.path.exists(path) and not FORCE_REBUILD_FEATURES:
        print(f"Loading cached {split_name} features from {path}")

        z = np.load(path, allow_pickle=True)

        return {
            "X": z["X"],
            "y_log": z["y_log"],
            "rv_in": z["rv_in"],
            "rv_fut": z["rv_fut"],
            "rv_last120": z["rv_last120"],
            "rv_scaled": z["rv_scaled"],
            "time_id": z["time_id"],
            "cluster_id": z["cluster_id"],
            "cluster_size": z["cluster_size"],
            "fit_ok": z["fit_ok"],
            "feature_names": list(z["feature_names"]),
        }

    tasks = build_feature_tasks(
        df=df,
        stock_cols=stock_cols,
        cluster_map=cluster_map,
        time_ids=time_ids,
        split_name=split_name,
        dev_n_time_ids=DEV_N_TIME_IDS,
    )

    print(
        f"  {split_name}: fitting ARMA-GARCH features "
        f"with n_jobs={n_jobs}"
    )

    start = time.perf_counter()

    if n_jobs == 1:
        rows = [
            extract_armagarch_feature_row(task, spec)
            for task in tasks
        ]

    else:
        rows = Parallel(
            n_jobs=n_jobs,
            backend="loky",
            verbose=5,
            batch_size=16,
        )(
            delayed(extract_armagarch_feature_row)(task, spec)
            for task in tasks
        )

    elapsed = (time.perf_counter() - start) / 60.0

    X = np.vstack([r["x"] for r in rows]).astype(np.float32)
    y_log = np.array([r["y_log"] for r in rows], dtype=np.float32)
    rv_in = np.array([r["rv_in"] for r in rows], dtype=np.float32)
    rv_fut = np.array([r["rv_fut"] for r in rows], dtype=np.float32)
    rv_last120 = np.array([r["rv_last120"] for r in rows], dtype=np.float32)
    rv_scaled = np.array([r["rv_scaled"] for r in rows], dtype=np.float32)
    time_id = np.array([r["time_id"] for r in rows], dtype=np.int32)
    cluster_id = np.array([r["cluster_id"] for r in rows], dtype=np.int32)
    cluster_size = np.array([r["cluster_size"] for r in rows], dtype=np.int32)
    fit_ok = np.array([r["fit_ok"] for r in rows], dtype=np.float32)

    print(
        f"  {split_name}: X={X.shape}, "
        f"fit success={fit_ok.mean() * 100:.1f}%, "
        f"time={elapsed:.2f} min"
    )

    np.savez_compressed(
        path,
        X=X,
        y_log=y_log,
        rv_in=rv_in,
        rv_fut=rv_fut,
        rv_last120=rv_last120,
        rv_scaled=rv_scaled,
        time_id=time_id,
        cluster_id=cluster_id,
        cluster_size=cluster_size,
        fit_ok=fit_ok,
        feature_names=np.array(feature_names, dtype=object),
    )

    print(f"  Saved {split_name} features to {path}")

    return {
        "X": X,
        "y_log": y_log,
        "rv_in": rv_in,
        "rv_fut": rv_fut,
        "rv_last120": rv_last120,
        "rv_scaled": rv_scaled,
        "time_id": time_id,
        "cluster_id": cluster_id,
        "cluster_size": cluster_size,
        "fit_ok": fit_ok,
        "feature_names": feature_names,
    }


In [21]:
# =============================================================================
# KNN
# =============================================================================

def fit_knn(
    X_train: np.ndarray,
    y_train: np.ndarray,
    k: int,
) -> tuple[StandardScaler, KNeighborsRegressor]:
    scaler = StandardScaler()

    Xs = scaler.fit_transform(X_train)

    model = KNeighborsRegressor(
        n_neighbors=min(k, len(X_train)),
        weights="distance",
        metric="minkowski",
        p=2,
        n_jobs=-1,
    )

    model.fit(Xs, y_train)

    return scaler, model


def predict_knn_log(
    scaler: StandardScaler,
    model: KNeighborsRegressor,
    X: np.ndarray,
) -> np.ndarray:
    Xs = scaler.transform(X)

    pred = model.predict(Xs).astype(np.float64)

    return np.clip(pred, -5.0, 5.0)


def cross_validate_knn(
    X: np.ndarray,
    y_log: np.ndarray,
    rv_in: np.ndarray,
    rv_fut: np.ndarray,
    cluster_size: np.ndarray,
    groups: np.ndarray,
    knn_ks: list[int],
    cv_folds: int = 5,
) -> tuple[int, pd.DataFrame]:
    unique_groups = np.unique(groups)
    n_splits = min(cv_folds, len(unique_groups))

    gkf = GroupKFold(n_splits=n_splits)

    rows = []

    print("\nKNN cross validation on train set")

    for k in knn_ks:
        fold_scores = []
        fold_weighted_scores = []
        fold_mse_scores = []
        fold_weighted_mse_scores = []
        fold_qlike_scores = []
        fold_weighted_qlike_scores = []

        for fold_i, (tr_idx, va_idx) in enumerate(
            gkf.split(X, y_log, groups),
            1,
        ):
            scaler, model = fit_knn(
                X_train=X[tr_idx],
                y_train=y_log[tr_idx],
                k=min(k, len(tr_idx)),
            )

            log_pred = predict_knn_log(
                scaler=scaler,
                model=model,
                X=X[va_idx],
            )

            rv_pred = np.exp(log_pred) * rv_in[va_idx]

            score = rmspe(
                y_true=rv_fut[va_idx],
                y_pred=rv_pred,
            )

            w_score = weighted_rmspe(
                y_true=rv_fut[va_idx],
                y_pred=rv_pred,
                weights=cluster_size[va_idx],
            )

            mse_score = mse(
                y_true=rv_fut[va_idx],
                y_pred=rv_pred,
            )

            wmse_score = weighted_mse(
                y_true=rv_fut[va_idx],
                y_pred=rv_pred,
                weights=cluster_size[va_idx],
            )

            qlike_score = qlike(
                y_true=rv_fut[va_idx],
                y_pred=rv_pred,
            )

            wqlike_score = weighted_qlike(
                y_true=rv_fut[va_idx],
                y_pred=rv_pred,
                weights=cluster_size[va_idx],
            )

            fold_scores.append(score)
            fold_weighted_scores.append(w_score)
            fold_mse_scores.append(mse_score)
            fold_weighted_mse_scores.append(wmse_score)
            fold_qlike_scores.append(qlike_score)
            fold_weighted_qlike_scores.append(wqlike_score)

        mean_score = float(np.mean(fold_scores))
        std_score = float(np.std(fold_scores))
        mean_w_score = float(np.mean(fold_weighted_scores))

        mean_mse = float(np.mean(fold_mse_scores))
        mean_w_mse = float(np.mean(fold_weighted_mse_scores))

        mean_qlike = float(np.mean(fold_qlike_scores))
        mean_w_qlike = float(np.mean(fold_weighted_qlike_scores))

        print(
            f"  k={k:<4} | "
            f"CV RMSPE={mean_score:.4f}% "
            f"+/- {std_score:.4f}% | "
            f"stock-weighted={mean_w_score:.4f}% | "
            f"MSE={mean_mse:.10f} | "
            f"stock-weighted MSE={mean_w_mse:.10f} | "
            f"QLIKE={mean_qlike:.10f} | "
            f"stock-weighted QLIKE={mean_w_qlike:.10f}"
        )

        rows.append(
            {
                "k": k,
                "cv_rmspe_mean": mean_score,
                "cv_rmspe_std": std_score,
                "cv_stock_weighted_rmspe_mean": mean_w_score,
                "cv_mse_mean": mean_mse,
                "cv_stock_weighted_mse_mean": mean_w_mse,
                "cv_qlike_mean": mean_qlike,
                "cv_stock_weighted_qlike_mean": mean_w_qlike,
            }
        )

    cv_df = pd.DataFrame(rows).sort_values("cv_stock_weighted_rmspe_mean")

    best_k = int(cv_df.iloc[0]["k"])

    print(f"\nBest KNN k by stock-weighted CV RMSPE: {best_k}")

    return best_k, cv_df


def make_oof_log_predictions(
    X: np.ndarray,
    y_log: np.ndarray,
    groups: np.ndarray,
    k: int,
    cv_folds: int = 5,
) -> np.ndarray:
    unique_groups = np.unique(groups)
    n_splits = min(cv_folds, len(unique_groups))

    gkf = GroupKFold(n_splits=n_splits)

    oof = np.zeros(len(y_log), dtype=np.float64)

    for tr_idx, va_idx in gkf.split(X, y_log, groups):
        scaler, model = fit_knn(
            X_train=X[tr_idx],
            y_train=y_log[tr_idx],
            k=min(k, len(tr_idx)),
        )

        oof[va_idx] = predict_knn_log(
            scaler=scaler,
            model=model,
            X=X[va_idx],
        )

    return np.clip(oof, -5.0, 5.0)


def compute_cluster_smearing(
    y_log: np.ndarray,
    oof_log_pred: np.ndarray,
    cluster_id: np.ndarray,
) -> dict[int, float]:
    smearing = {}

    for cid in sorted(np.unique(cluster_id)):
        mask = cluster_id == cid

        resid = y_log[mask] - oof_log_pred[mask]
        resid = np.clip(resid, -5.0, 5.0)

        factor = float(np.mean(np.exp(resid)))

        if not np.isfinite(factor) or factor <= 0:
            factor = 1.0

        factor = float(np.clip(factor, 0.25, 4.0))

        smearing[int(cid)] = factor

    return smearing


def evaluate_predictions(
    label: str,
    data: dict,
    log_pred: np.ndarray,
    smearing: Optional[dict[int, float]] = None,
) -> dict:
    rv_pred_raw = np.exp(np.clip(log_pred, -5.0, 5.0)) * data["rv_in"]

    raw_rmspe = rmspe(data["rv_fut"], rv_pred_raw)
    raw_wrmspe = weighted_rmspe(
        data["rv_fut"],
        rv_pred_raw,
        data["cluster_size"],
    )
    raw_mse = mse(data["rv_fut"], rv_pred_raw)
    raw_wmse = weighted_mse(
        data["rv_fut"],
        rv_pred_raw,
        data["cluster_size"],
    )
    raw_qlike = qlike(data["rv_fut"], rv_pred_raw)
    raw_wqlike = weighted_qlike(
        data["rv_fut"],
        rv_pred_raw,
        data["cluster_size"],
    )

    last_rmspe = rmspe(data["rv_fut"], data["rv_last120"])
    last_wrmspe = weighted_rmspe(
        data["rv_fut"],
        data["rv_last120"],
        data["cluster_size"],
    )
    last_mse = mse(data["rv_fut"], data["rv_last120"])
    last_wmse = weighted_mse(
        data["rv_fut"],
        data["rv_last120"],
        data["cluster_size"],
    )
    last_qlike = qlike(data["rv_fut"], data["rv_last120"])
    last_wqlike = weighted_qlike(
        data["rv_fut"],
        data["rv_last120"],
        data["cluster_size"],
    )

    scaled_rmspe = rmspe(data["rv_fut"], data["rv_scaled"])
    scaled_wrmspe = weighted_rmspe(
        data["rv_fut"],
        data["rv_scaled"],
        data["cluster_size"],
    )
    scaled_mse = mse(data["rv_fut"], data["rv_scaled"])
    scaled_wmse = weighted_mse(
        data["rv_fut"],
        data["rv_scaled"],
        data["cluster_size"],
    )
    scaled_qlike = qlike(data["rv_fut"], data["rv_scaled"])
    scaled_wqlike = weighted_qlike(
        data["rv_fut"],
        data["rv_scaled"],
        data["cluster_size"],
    )

    print(f"\n{label}")

    print("\n  KNN raw")
    print(f"    RMSPE                  : {raw_rmspe:.4f}%")
    print(f"    Stock-weighted RMSPE   : {raw_wrmspe:.4f}%")
    print(f"    MSE                    : {raw_mse:.10f}")
    print(f"    Stock-weighted MSE     : {raw_wmse:.10f}")
    print(f"    QLIKE                  : {raw_qlike:.10f}")
    print(f"    Stock-weighted QLIKE   : {raw_wqlike:.10f}")

    print("\n  Last-120 baseline")
    print(f"    RMSPE                  : {last_rmspe:.4f}%")
    print(f"    Stock-weighted RMSPE   : {last_wrmspe:.4f}%")
    print(f"    MSE                    : {last_mse:.10f}")
    print(f"    Stock-weighted MSE     : {last_wmse:.10f}")
    print(f"    QLIKE                  : {last_qlike:.10f}")
    print(f"    Stock-weighted QLIKE   : {last_wqlike:.10f}")

    print("\n  Scaled-input baseline")
    print(f"    RMSPE                  : {scaled_rmspe:.4f}%")
    print(f"    Stock-weighted RMSPE   : {scaled_wrmspe:.4f}%")
    print(f"    MSE                    : {scaled_mse:.10f}")
    print(f"    Stock-weighted MSE     : {scaled_wmse:.10f}")
    print(f"    QLIKE                  : {scaled_qlike:.10f}")
    print(f"    Stock-weighted QLIKE   : {scaled_wqlike:.10f}")

    out = {
        "raw_rmspe": raw_rmspe,
        "raw_stock_weighted_rmspe": raw_wrmspe,
        "raw_mse": raw_mse,
        "raw_stock_weighted_mse": raw_wmse,
        "raw_qlike": raw_qlike,
        "raw_stock_weighted_qlike": raw_wqlike,

        "last_rmspe": last_rmspe,
        "last_stock_weighted_rmspe": last_wrmspe,
        "last_mse": last_mse,
        "last_stock_weighted_mse": last_wmse,
        "last_qlike": last_qlike,
        "last_stock_weighted_qlike": last_wqlike,

        "scaled_rmspe": scaled_rmspe,
        "scaled_stock_weighted_rmspe": scaled_wrmspe,
        "scaled_mse": scaled_mse,
        "scaled_stock_weighted_mse": scaled_wmse,
        "scaled_qlike": scaled_qlike,
        "scaled_stock_weighted_qlike": scaled_wqlike,
    }

    if smearing is not None:
        smear_arr = np.array(
            [smearing.get(int(cid), 1.0) for cid in data["cluster_id"]],
            dtype=np.float64,
        )

        rv_pred_smear = rv_pred_raw * smear_arr

        smear_rmspe = rmspe(data["rv_fut"], rv_pred_smear)
        smear_wrmspe = weighted_rmspe(
            data["rv_fut"],
            rv_pred_smear,
            data["cluster_size"],
        )
        smear_mse = mse(data["rv_fut"], rv_pred_smear)
        smear_wmse = weighted_mse(
            data["rv_fut"],
            rv_pred_smear,
            data["cluster_size"],
        )
        smear_qlike = qlike(data["rv_fut"], rv_pred_smear)
        smear_wqlike = weighted_qlike(
            data["rv_fut"],
            rv_pred_smear,
            data["cluster_size"],
        )

        print("\n  KNN smearing")
        print(f"    RMSPE                  : {smear_rmspe:.4f}%")
        print(f"    Stock-weighted RMSPE   : {smear_wrmspe:.4f}%")
        print(f"    MSE                    : {smear_mse:.10f}")
        print(f"    Stock-weighted MSE     : {smear_wmse:.10f}")
        print(f"    QLIKE                  : {smear_qlike:.10f}")
        print(f"    Stock-weighted QLIKE   : {smear_wqlike:.10f}")

        out["smearing_rmspe"] = smear_rmspe
        out["smearing_stock_weighted_rmspe"] = smear_wrmspe
        out["smearing_mse"] = smear_mse
        out["smearing_stock_weighted_mse"] = smear_wmse
        out["smearing_qlike"] = smear_qlike
        out["smearing_stock_weighted_qlike"] = smear_wqlike

        print("\n  Per-cluster smearing scores")

        for cid in sorted(np.unique(data["cluster_id"])):
            mask = data["cluster_id"] == cid

            c_pred = rv_pred_smear[mask]

            c_score = rmspe(
                y_true=data["rv_fut"][mask],
                y_pred=c_pred,
            )

            c_wscore = weighted_rmspe(
                y_true=data["rv_fut"][mask],
                y_pred=c_pred,
                weights=data["cluster_size"][mask],
            )

            c_mse = mse(
                y_true=data["rv_fut"][mask],
                y_pred=c_pred,
            )

            c_qlike = qlike(
                y_true=data["rv_fut"][mask],
                y_pred=c_pred,
            )

            print(
                f"    Cluster {int(cid):<3} | "
                f"n={mask.sum():<5} | "
                f"RMSPE={c_score:.4f}% | "
                f"weighted={c_wscore:.4f}% | "
                f"MSE={c_mse:.10f} | "
                f"QLIKE={c_qlike:.10f} | "
                f"smear={smearing.get(int(cid), 1.0):.4f}"
            )

    return out


def fit_and_predict(
    train_data: dict,
    eval_data: dict,
    k: int,
) -> tuple[np.ndarray, StandardScaler, KNeighborsRegressor]:
    scaler, model = fit_knn(
        X_train=train_data["X"],
        y_train=train_data["y_log"],
        k=k,
    )

    log_pred = predict_knn_log(
        scaler=scaler,
        model=model,
        X=eval_data["X"],
    )

    return log_pred, scaler, model


In [22]:
# =============================================================================
# PIPELINE
# =============================================================================

def run_clustered_armagarch_knn(
    df: pd.DataFrame,
    n_clusters: int = 10,
    spec: dict = SPEC,
    val_ratio: float = 0.15,
    test_ratio: float = 0.15,
    seed: int = 42,
    knn_ks: list[int] = KNN_KS,
    cv_folds: int = 5,
    n_jobs: int = 1,
) -> dict:
    start_all = time.perf_counter()

    print("=" * 80)
    print("Clustered ARMA-GARCH feature KNN")
    print(
        f"ARMA({spec['ar']},{spec['ma']})-"
        f"GARCH({spec['p']},{spec['q']}) | "
        f"dist={spec['dist']}"
    )
    print("=" * 80)

    stock_cols = [
        c for c in df.columns
        if c not in ("time_id", "seconds_in_bucket")
    ]

    print(f"Rows: {len(df):,}")
    print(f"Stocks: {len(stock_cols)}")
    print(f"Time IDs: {df['time_id'].nunique():,}")
    print(f"Input window: 0 to {INPUT_LEN - 1}")
    print(f"Target window: {INPUT_LEN} to {TOTAL_LEN - 1}")
    print(f"Clusters: {n_clusters}")
    print(f"N_JOBS: {n_jobs}")

    splits = make_splits(
        df=df,
        val_ratio=val_ratio,
        test_ratio=test_ratio,
        seed=seed,
    )

    train_ids = splits["split_summary"]["train"]["ids"]
    val_ids = splits["split_summary"]["val"]["ids"]
    test_ids = splits["split_summary"]["test"]["ids"]

    print(
        f"Split: train={len(train_ids):,}, "
        f"val={len(val_ids):,}, "
        f"test={len(test_ids):,}"
    )

    if DEV_N_TIME_IDS is not None:
        print(f"DEV mode: using first {DEV_N_TIME_IDS} time_ids per split")

    profiles, cluster_map = get_or_build_cluster_map(
        df=df,
        stock_cols=stock_cols,
        train_ids=train_ids,
        n_clusters=n_clusters,
        seed=seed,
    )

    expected_fits = (
        len(train_ids) + len(val_ids) + len(test_ids)
    ) * n_clusters

    if DEV_N_TIME_IDS is not None:
        expected_fits = (
            min(DEV_N_TIME_IDS, len(train_ids))
            + min(DEV_N_TIME_IDS, len(val_ids))
            + min(DEV_N_TIME_IDS, len(test_ids))
        ) * n_clusters

    print(f"Expected ARMA-GARCH fits if cache is missing: {expected_fits:,}")

    train_data = build_or_load_features(
        df=df,
        stock_cols=stock_cols,
        cluster_map=cluster_map,
        time_ids=train_ids,
        split_name="train",
        spec=spec,
        n_jobs=n_jobs,
    )

    val_data = build_or_load_features(
        df=df,
        stock_cols=stock_cols,
        cluster_map=cluster_map,
        time_ids=val_ids,
        split_name="val",
        spec=spec,
        n_jobs=n_jobs,
    )

    test_data = build_or_load_features(
        df=df,
        stock_cols=stock_cols,
        cluster_map=cluster_map,
        time_ids=test_ids,
        split_name="test",
        spec=spec,
        n_jobs=n_jobs,
    )

    best_k, cv_df = cross_validate_knn(
        X=train_data["X"],
        y_log=train_data["y_log"],
        rv_in=train_data["rv_in"],
        rv_fut=train_data["rv_fut"],
        cluster_size=train_data["cluster_size"],
        groups=train_data["time_id"],
        knn_ks=knn_ks,
        cv_folds=cv_folds,
    )

    print("\nBuilding train-only smearing factors")

    train_oof = make_oof_log_predictions(
        X=train_data["X"],
        y_log=train_data["y_log"],
        groups=train_data["time_id"],
        k=best_k,
        cv_folds=cv_folds,
    )

    train_smearing = compute_cluster_smearing(
        y_log=train_data["y_log"],
        oof_log_pred=train_oof,
        cluster_id=train_data["cluster_id"],
    )

    val_log_pred, _, _ = fit_and_predict(
        train_data=train_data,
        eval_data=val_data,
        k=best_k,
    )

    val_metrics = evaluate_predictions(
        label="Validation evaluation",
        data=val_data,
        log_pred=val_log_pred,
        smearing=train_smearing,
    )

    print("\nFitting final KNN on train plus validation")

    train_val_data = {
        "X": np.vstack([train_data["X"], val_data["X"]]),
        "y_log": np.concatenate([train_data["y_log"], val_data["y_log"]]),
        "rv_in": np.concatenate([train_data["rv_in"], val_data["rv_in"]]),
        "rv_fut": np.concatenate([train_data["rv_fut"], val_data["rv_fut"]]),
        "rv_last120": np.concatenate(
            [train_data["rv_last120"], val_data["rv_last120"]]
        ),
        "rv_scaled": np.concatenate(
            [train_data["rv_scaled"], val_data["rv_scaled"]]
        ),
        "time_id": np.concatenate([train_data["time_id"], val_data["time_id"]]),
        "cluster_id": np.concatenate(
            [train_data["cluster_id"], val_data["cluster_id"]]
        ),
        "cluster_size": np.concatenate(
            [train_data["cluster_size"], val_data["cluster_size"]]
        ),
    }

    train_val_oof = make_oof_log_predictions(
        X=train_val_data["X"],
        y_log=train_val_data["y_log"],
        groups=train_val_data["time_id"],
        k=best_k,
        cv_folds=cv_folds,
    )

    train_val_smearing = compute_cluster_smearing(
        y_log=train_val_data["y_log"],
        oof_log_pred=train_val_oof,
        cluster_id=train_val_data["cluster_id"],
    )

    test_log_pred, final_scaler, final_model = fit_and_predict(
        train_data=train_val_data,
        eval_data=test_data,
        k=best_k,
    )

    test_metrics = evaluate_predictions(
        label="Final test evaluation",
        data=test_data,
        log_pred=test_log_pred,
        smearing=train_val_smearing,
    )

    total_min = (time.perf_counter() - start_all) / 60.0

    print("\n" + "=" * 80)
    print("Final summary")
    print("=" * 80)
    print(f"Best KNN k: {best_k}")

    print(f"Validation raw stock-weighted RMSPE: {val_metrics['raw_stock_weighted_rmspe']:.4f}%")
    print(f"Validation raw stock-weighted MSE  : {val_metrics['raw_stock_weighted_mse']:.10f}")
    print(f"Validation raw stock-weighted QLIKE: {val_metrics['raw_stock_weighted_qlike']:.10f}")

    print(f"Validation smearing stock-weighted RMSPE: {val_metrics['smearing_stock_weighted_rmspe']:.4f}%")
    print(f"Validation smearing stock-weighted MSE  : {val_metrics['smearing_stock_weighted_mse']:.10f}")
    print(f"Validation smearing stock-weighted QLIKE: {val_metrics['smearing_stock_weighted_qlike']:.10f}")

    print(f"Test raw stock-weighted RMSPE: {test_metrics['raw_stock_weighted_rmspe']:.4f}%")
    print(f"Test raw stock-weighted MSE  : {test_metrics['raw_stock_weighted_mse']:.10f}")
    print(f"Test raw stock-weighted QLIKE: {test_metrics['raw_stock_weighted_qlike']:.10f}")

    print(f"Test smearing stock-weighted RMSPE: {test_metrics['smearing_stock_weighted_rmspe']:.4f}%")
    print(f"Test smearing stock-weighted MSE  : {test_metrics['smearing_stock_weighted_mse']:.10f}")
    print(f"Test smearing stock-weighted QLIKE: {test_metrics['smearing_stock_weighted_qlike']:.10f}")

    print(f"Test last-120 stock-weighted RMSPE: {test_metrics['last_stock_weighted_rmspe']:.4f}%")
    print(f"Test last-120 stock-weighted MSE  : {test_metrics['last_stock_weighted_mse']:.10f}")
    print(f"Test last-120 stock-weighted QLIKE: {test_metrics['last_stock_weighted_qlike']:.10f}")

    print(f"Test scaled-input stock-weighted RMSPE: {test_metrics['scaled_stock_weighted_rmspe']:.4f}%")
    print(f"Test scaled-input stock-weighted MSE  : {test_metrics['scaled_stock_weighted_mse']:.10f}")
    print(f"Test scaled-input stock-weighted QLIKE: {test_metrics['scaled_stock_weighted_qlike']:.10f}")

    print(f"Total runtime: {total_min:.2f} minutes")
    print("=" * 80)

    return {
        "splits": splits,
        "profiles": profiles,
        "cluster_map": cluster_map,
        "feature_names": train_data["feature_names"],
        "train_data": train_data,
        "val_data": val_data,
        "test_data": test_data,
        "cv_df": cv_df,
        "best_k": best_k,
        "train_smearing": train_smearing,
        "train_val_smearing": train_val_smearing,
        "val_metrics": val_metrics,
        "test_metrics": test_metrics,
        "final_scaler": final_scaler,
        "final_model": final_model,
        "total_minutes": total_min,
    }

In [23]:

# =============================================================================
# LOADING THE DATA
# =============================================================================

def load_final_csv_chunked(
    path: str = "final.csv",
    chunksize: int = 100_000,
) -> pd.DataFrame:
    header = pd.read_csv(path, nrows=0)

    stock_cols = [
        c for c in header.columns
        if c not in ("time_id", "seconds_in_bucket")
    ]

    dtype_map = {
        "time_id": np.int32,
        "seconds_in_bucket": np.int16,
        **{c: np.float32 for c in stock_cols},
    }

    chunks = []

    print(f"Reading {path} in chunks")

    for i, chunk in enumerate(
        pd.read_csv(
            path,
            dtype=dtype_map,
            chunksize=chunksize,
        )
    ):
        chunks.append(chunk)

        if i % 10 == 0:
            print(f"  chunk {i}, rows so far {(i + 1) * chunksize:,}")

    df = pd.concat(chunks, ignore_index=True)

    print(
        f"Loaded CSV: {df.shape} | "
        f"memory={df.memory_usage(deep=True).sum() / 1e6:.0f} MB"
    )

    return df

In [24]:
if __name__ == "__main__":
    if os.path.exists("final.parquet"):
        print("Loading from final.parquet")

        df = pd.read_parquet("final.parquet")

        print(
            f"Loaded: {df.shape} | "
            f"memory={df.memory_usage(deep=True).sum() / 1e6:.0f} MB"
        )

    else:
        print("Loading final.csv and saving final.parquet")

        df = load_final_csv_chunked(
            path="final.csv",
            chunksize=100_000,
        )

        df.to_parquet(
            "final.parquet",
            index=False,
        )

        print("Saved final.parquet")

    results = run_clustered_armagarch_knn(
        df=df,
        n_clusters=N_CLUSTERS,
        spec=SPEC,
        val_ratio=VAL_RATIO,
        test_ratio=TEST_RATIO,
        seed=SEED,
        knn_ks=KNN_KS,
        cv_folds=CV_FOLDS,
        n_jobs=N_JOBS,
    )

Loading from final.parquet
Loaded: (2298000, 114) | memory=1043 MB
Clustered ARMA-GARCH feature KNN
ARMA(1,1)-GARCH(1,1) | dist=normal
Rows: 2,298,000
Stocks: 112
Time IDs: 3,830
Input window: 0 to 479
Target window: 480 to 599
Clusters: 10
N_JOBS: 8
Split: train=2,682, val=574, test=574
Loaded cached profiles and cluster map
  Cluster sizes: {0: 21, 1: 17, 2: 2, 3: 1, 4: 12, 5: 16, 6: 21, 7: 4, 8: 4, 9: 14}
Expected ARMA-GARCH fits if cache is missing: 38,300
Loading cached train features from arma_garch_knn_cache\features_train_full_k10_seed42_arma11_garch11_normal.npz
Loading cached val features from arma_garch_knn_cache\features_val_full_k10_seed42_arma11_garch11_normal.npz
Loading cached test features from arma_garch_knn_cache\features_test_full_k10_seed42_arma11_garch11_normal.npz

KNN cross validation on train set
  k=3    | CV RMSPE=53.8933% +/- 4.4219% | stock-weighted=20.0979% | MSE=0.0000003221 | stock-weighted MSE=0.0000001085 | QLIKE=0.0394203003 | stock-weighted QLIKE=0.0